# CFIHOS Processing

This notebook contains the code to parse the CFIHOS specificiation (both 1.5.1 and 2.0 is supported) of tag class, tag class properties, property and uom (unit of measures).

**To start off**:

- Choose the appropriate version of CFIHOS you would like to parse. You do this by setting the `SELECTED` to either either `cfihos_version_151` or `cfihos_version_20`.
- Choose the name of the output file in `output_file_name`.
- Run the selection of cells within "Read Source CFIHOS Data"
  - Optionally, run the selection of cells within "Integrate with Aveva ISM Class Library" to also include the class library's Presence value to any attribute mapped to a CFIHOS property*.

*By default the Presence is set to "Not Applicable" if the integration with a third party class library is not executed.

In [ ]:
import os
from pathlib import Path

import duckdb
import polars as pl
import polars.selectors as cs
from classes.cfihos import CfihosClass, Property
from rich import print as rprint

In [4]:
folder = Path(os.path.abspath("")) / "CFIHOS"


cfihos_version_151 = "CFIHOS V1.5.1 excel format v1.5.1.xlsx"
cfihos_version_20 = "CORE-CFIHOS-V2.0-excel-FINAL.xlsx"

cfihos_path = folder / cfihos_version_151
cfihos_2_path = folder / cfihos_version_20

SELECTED = cfihos_version_151   # select either cfihos_version_151 or cfihos_version_20

In [5]:
# just name of the file, excluding file extension
output_file_name = "cfihos_classes"

# Read Source CFIHOS Data

In [6]:
PATH = folder / SELECTED

path_sheet_map = {
    cfihos_path: {
        "all_classes": "tag class",
        "all_class_properties": "tag class properties",
        "all_properties": "property",
        "uom": "unit of measure dimension",
    },
    cfihos_2_path: {
        "all_classes": "tag class",
        "all_class_properties": "tag class property",
        "all_properties": "property",
        "uom": "unit of measure",
    },
}

In [7]:
all_classes = pl.read_excel(PATH, sheet_name=path_sheet_map[PATH]["all_classes"])
all_class_properties = pl.read_excel(PATH, sheet_name=path_sheet_map[PATH]["all_class_properties"])
all_properties = pl.read_excel(PATH, sheet_name=path_sheet_map[PATH]["all_properties"])
uom_dimensions = pl.read_excel(PATH, sheet_name=path_sheet_map[PATH]["uom"])

Could not determine dtype for column 5, falling back to string
Could not determine dtype for column 7, falling back to string
Could not determine dtype for column 7, falling back to string
Could not determine dtype for column 8, falling back to string
Could not determine dtype for column 7, falling back to string
Could not determine dtype for column 3, falling back to string
Could not determine dtype for column 6, falling back to string


In [8]:
# swap whitespace for underscores in all column names
all_classes.columns = [col.replace(" ", "_") for col in all_classes.columns]
all_class_properties.columns = [col.replace(" ", "_") for col in all_class_properties.columns]
all_properties.columns = [col.replace(" ", "_") for col in all_properties.columns]
uom_dimensions.columns = [col.replace(" ", "_") for col in uom_dimensions.columns]

Include the unit of measure dimension name as part of the properties, to avoid having to deal with uom_dimensions dataframe later.

In [9]:
all_properties = all_properties.join(
    uom_dimensions["unit_of_measure_dimension_code", "unit_of_measure_dimension_name"].unique(),
    how="left",
    on="unit_of_measure_dimension_code",
    validate="m:1",
)

## Create a level column to present the depth in the CFIHOS tag class hierarchy

In [10]:
all_classes = all_classes.with_columns(pl.col("parent_tag_class_name").replace("", None))
df_duckdb = duckdb.from_df(all_classes.to_pandas())


# Use a recursive CTE to assign levels
query = """
WITH RECURSIVE hierarchy AS (
    SELECT tag_class_name, parent_tag_class_name, 0 AS level
    FROM df_duckdb
    WHERE parent_tag_class_name IS NULL
    UNION ALL
    SELECT a.tag_class_name, a.parent_tag_class_name, h.level + 1
    FROM df_duckdb a
    JOIN hierarchy h ON a.parent_tag_class_name = h.tag_class_name
)
SELECT * FROM hierarchy
"""

result = duckdb.query(query).to_df()

result_polars = pl.from_pandas(result)

all_classes = all_classes.join(result_polars["tag_class_name", "level"], on="tag_class_name", how="left")

## Simplified Object-Oriented Representation of CFIHOS Tag Classes

Every parent tag class ought to be represented as an instance of `CfihosClass`. For every parent tag class, we store their related children tag classes by both name and id. 

The output will look like this for one CfihosClass, here we are using Conveyor as an example:

```json
{
    "CFIHOS_unique_id": "CFIHOS-30000564",
    "children_by_id": [
      "CFIHOS-30000570",
      "CFIHOS-30000563",
      "CFIHOS-30000226",
      "CFIHOS-30000562",
      "CFIHOS-30000569"
    ],
    "children_by_name": [
      "belt conveyor",
      "chain conveyor",
      "roller conveyor",
      "screw conveyor",
      "slat conveyor"
    ],
    "level": 2,
    "parent_by_id": "CFIHOS-30000314",
    "parent_by_name": "mechanical equipment class",
    "properties": {...},
    "reason_for_having_children": [
      ""
    ],
    "reason_for_having_class": "",
    "tag_class_definition": "A physical object for moving articles or bulk material from place to place (as by an endless moving belt or a chain of receptacles).",
    "tag_class_name": "conveyor"    
}
```

A property will have the following structure:
```json
{
    "EXPLOSION PROTECTION ZONE": {
        "presence": "Preferred",
        "property_data_type": "Text",
        "property_definition": "Specify the hazardous zone in which the functional or physical object is expected to operate.",
        "property_name": "explosion protection zone",
        "tag_property_CFIHOS_unique_id": "CFIHOS-40000113"
    },
}
```

Keep in mind that the `presence` attribute is derived from the integration done to the example project Class Library.

The following will explain where the property gets its values from. There may be some minor differences in the column names between CFIHOS 1.5.1. and CFIHOS 2.0.. For the CfihosClass:

 - `CFIHOS_unique_id`: "CFIHOS unique id" from 'tag class' sheet in CFIHOS 1.5.1 / "CFIHOS unique code" from 'tag class' sheet in CFIHOS 2.0.
 - `children_by_id`: a list of tag classes that has this tag class as parent or ancestor, via the "parent tag class name" relation in the 'tag class' sheet.
 - `children_by_name`: a list of tag classes that has this tag class as parent or ancestor, via the "parent tag class name" relation in the 'tag class' sheet.
 - `level`: the depth in which the tag class resides in, in the CFIHOS tag class hierarchy, starting from **class** (CFIHOS-30000311) at level 0.
 - `parent_by_id`: The CFIHOS unique id of the parent tag class
 - `parent_by_name`: The 'parent tag class name' of the particular CFIHOS tag class   
 - `reason_for_having_children`: collection of all children's 'reason for having class' (1.5.1) / 'tag class existence reason description' (2.0).
 - `reason_for_having_class`: the tag class' 'reason for having class' (1.5.1) / 'tag class existence reason description' (2.0).
 - `tag_class_definition`: the 'tag class definition' of the tag class.
 - `tag_class_name`: the 'tag class name'.

For a Property:

 - `presence`: 'Presence' column from the "ISM Attributes" / "ISM Functional Class Attributes"
 - `property_data_type`: 'property data type' column from the "Property" sheet in CFIHOS 1.5.1. and 2.0.
 - `property_definition`: 'property definition' column from the "Property" sheet in CFIHOS 1.5.1. and 2.0.
 - `property_name`: 'property name' column from the "Property" sheet in CFIHOS 1.5.1. and 2.0.
 - `tag_property_CFIHOS_unique_id`: 'tag property CFIHOS unique id' / 'tag property CFIHOS unique code' column from the "tag class properties" from CFIHOS 1.5.1 and 2.0 respectively.
 

## Property Propagation

Many high level tag classes don't have any properties related to them. However, as we have the desire to represent a high level tag class, like mechanical equipment class, as a bucket for any tag class that is a child of mechanical equipment class. This means we must ensure that all unique properties for any child of mechanical equipment class is also represented in the mechanical equipment class itself. 

As an example, CFIHOS-40000460 (upper limit design product outflow rate) is only defined for the **tank** (CFIHOS-30000593) tag class. The tank's parent is the mechanical equipment class. So for the mechanical equipment class to be able to represent an instance of a tank appropriately, it needs to also contain cover the property that is only defined for the tank tag class.

In [11]:
PROPERTY_PROPAGATION = False

## Selection of Parent Tag Classes

We filter away TERMINATED classes as they do not have any properties related to them within CFIHOS. We do this by filtering for tag classes where the `status` is "STANDARD". This only applies to CFIHOS 1.5.1.

- The status of TERMINATED is only present in CFIHOS 1.5.1. Hence we try to catch the ColumnNotFoundError in case version 2.0 is parsed. At the time of writing the 2.0 is the latest version, thus naturally contains all the non-terminated tag classes.

In [12]:
try:
    parent_tag_class_names = (
        all_classes.filter(pl.col("status") == "STANDARD")["parent_tag_class_name"].unique().to_list()
    )
except pl.exceptions.ColumnNotFoundError:
    parent_tag_class_names = all_classes["parent_tag_class_name"].unique().to_list()

In [13]:
count_all_tag_classes = all_classes.select(cs.matches("tag_class_name")).height

print(f"Total number of tag classes: {count_all_tag_classes}")

Total number of tag classes: 935


In [14]:
count_parent_tag_classes = (
    all_classes.select(cs.matches("tag_class_name") & cs.matches("parent_tag_class_name")).unique().height
)

print(f"Number of parent tag classes: {count_parent_tag_classes}")

Number of parent tag classes: 93


Creating a column map so that we can support both CFIHOS versions without having too much logic to handle that in the parsing itself.

The structure is as follows:
```json
{
    "cfihos-version": {
        "dataframe name": {
            "standardized column name": "actual column name from the spreadsheet"
        }
    }
}
```

In [15]:
COL_MAP = {
    cfihos_path: {
        "all_classes": {
            "parent_tag_class_name": "parent_tag_class_name",
            "tag_class_cfihos_unique_id": "CFIHOS_unique_id",
            "reason_for_having_class": "reason_for_having_class",
        },
        "all_class_properties": {
            "tag_class_cfihos_unique_id": "tag_class_CFIHOS_unique_id",
            "property_cfihos_unique_id": "tag_property_CFIHOS_unique_id",
            "property_name": "tag_property_name",
        },
        "all_properties": {
            "cfihos_id": "CFIHOS_unique_id",
        },
    },
    cfihos_2_path: {
        "all_classes": {
            "parent_tag_class_name": "parent_tag_class_name",
            "tag_class_cfihos_unique_id": "CFIHOS_unique_code",
            "reason_for_having_class": "tag_class_existence_reason_description",
        },
        "all_class_properties": {
            "tag_class_cfihos_unique_id": "tag_class_CFIHOS_unique_code",
            "property_cfihos_unique_id": "property_CFIHOS_unique_code",
            "property_name": "property_name",
        },
        "all_properties": {
            "cfihos_id": "CFIHOS_unique_code",
        },
    },
}

## Parsing

In [16]:
def children_selection(
    tag_class: pl.DataFrame, parent_tag_class_name: str | list[str], id_col: str, bg_col: str, done: bool = False
) -> tuple[list, list, list]:
    """Select children of a tag class.

    Args:
        tag_class (pl.DataFrame): The tag class dataframe from CFIHOS.
        parent_tag_class_name (str | list[str]): The name of the parent tag class or classes.
        id_col (str): The column name for the tag class CFIHOS ID.
        bg_col (str): The column name for the reason for having the class.
        done (bool): A flag indicating whether the recursion is complete.

    Returns:
        tuple[list, list, list]: A tuple containing:
            - children_ids: A list of unique CFIHOS IDs of the children tag classes.
            - children_names: A list of unique names of the children tag classes.
            - children_background: A list of unique reasons for having the children tag classes.
            - done: A boolean indicating whether the recursion is complete.
    """
    if isinstance(parent_tag_class_name, str):
        parent_tag_class_name = [parent_tag_class_name]

    children = tag_class.filter(pl.col("parent_tag_class_name").is_in(parent_tag_class_name))
    children_ids = children[id_col].unique().to_list()
    children_names = children["tag_class_name"].unique().to_list()
    children_background = children.filter(pl.col(bg_col).is_not_null())[bg_col].unique().to_list()

    done = False
    while not done:
        if not children_names:
            done = True
            break
        sub_children_ids, sub_children_names, sub_children_background, done = children_selection(
            tag_class, children_names, id_col, bg_col
        )
        children_ids.extend(sub_children_ids)
        children_names.extend(sub_children_names)
        children_background.extend(sub_children_background)
    return children_ids, children_names, children_background, done

In [17]:
# column names matched for the respective CFIHOS version

# NOTE: variables are prefixed in accordance with the CFIHOS sheets
# tag_class_* for tag class, tag_class_prop_* for tag class properties, prop_* for properties

tag_class_cfihos_id_col = COL_MAP[PATH]["all_classes"]["tag_class_cfihos_unique_id"]
tag_class_background_col = COL_MAP[PATH]["all_classes"]["reason_for_having_class"]
tag_class_parent_col = COL_MAP[PATH]["all_classes"]["parent_tag_class_name"]

tag_class_prop_cfihos_unique_id_col = COL_MAP[PATH]["all_class_properties"]["tag_class_cfihos_unique_id"]
tag_class_prop_property_cfihos_unique_id_col = COL_MAP[PATH]["all_class_properties"]["property_cfihos_unique_id"]
tag_class_prop_property_name_col = COL_MAP[PATH]["all_class_properties"]["property_name"]

prop_cfihos_unique_id_col = COL_MAP[PATH]["all_properties"]["cfihos_id"]

In [18]:
try:
    tag_class_names = all_classes.filter(pl.col("status") == "STANDARD")["tag_class_name"].unique().to_list()
except pl.exceptions.ColumnNotFoundError:
    # CFIHOS 2.0. does not have a status column
    tag_class_names = all_classes["tag_class_name"].unique().to_list()

In [19]:
cfihos_classes: list[CfihosClass] = []
for tag_class in tag_class_names:
    try:
        # lookup each parent tag class by the `tag_class_name` column, that way we can also find their parent class
        self_ = all_classes.row(by_predicate=(pl.col("tag_class_name") == tag_class), named=True)
    except pl.exceptions.NoRowsReturnedError:
        display(f"Skipping '{tag_class}' because it has no self")
        continue

    parent_name = self_[tag_class_parent_col]
    if parent_name is None or self_["level"] == 0:
        parent_tag_class_code = ""
    else:
        try:
            parent_tag_class_code = all_classes.row(
                by_predicate=(pl.col("tag_class_name") == parent_name), named=True
            )[tag_class_cfihos_id_col]
        except pl.exceptions.NoRowsReturnedError as e:
            raise e

    children_ids, children_names, children_background, _ = children_selection(
        all_classes, tag_class, tag_class_cfihos_id_col, tag_class_background_col
    )

    # include the tag class itself in the list, however abstract classes does not appear to have properties
    # NOTE: the inclusion of `*children_ids` is what facilitates the property propagation from child to parent
    if PROPERTY_PROPAGATION:
        class_filter = [*children_ids, self_[tag_class_cfihos_id_col]]
    else:
        class_filter = [self_[tag_class_cfihos_id_col]]

    class_properties = all_class_properties.filter(pl.col(tag_class_prop_cfihos_unique_id_col).is_in(class_filter))[
        "tag_class_name",
        tag_class_prop_cfihos_unique_id_col,
        tag_class_prop_property_cfihos_unique_id_col,
        tag_class_prop_property_name_col,
    ]

    properties = class_properties.join(
        all_properties[
            "property_name",
            "property_definition",
            "property_data_type",
            "unit_of_measure_dimension_code",
            "unit_of_measure_dimension_name",
            prop_cfihos_unique_id_col,
        ],
        left_on=tag_class_prop_property_cfihos_unique_id_col,
        right_on=prop_cfihos_unique_id_col,
    )

    # convert each property to a Property object
    properties = {
        row["property_name"].upper(): Property.model_validate(row) for row in properties.iter_rows(named=True)
    }
    cfihos_class = CfihosClass(
        tag_class_name=tag_class,
        CFIHOS_unique_id=self_[tag_class_cfihos_id_col],
        tag_class_definition=self_["tag_class_definition"],
        level=self_["level"],
        reason_for_having_class=self_[tag_class_background_col],
        properties=properties,
        children_by_name=children_names,
        children_by_id=children_ids,
        reason_for_having_children=children_background,
        parent_by_name=self_[tag_class_parent_col],
        parent_by_id=parent_tag_class_code,
    )

    cfihos_classes.append(cfihos_class)

C:\Users\JanIngeBergseth\AppData\Local\Temp\ipykernel_8188\1781880281.py:12: UserWarning: Comparisons with None always result in null. Consider using `.is_null()` or `.is_not_null()`.
  by_predicate=(pl.col("tag_class_name") == self_["parent_tag_class_name"]), named=True


In [20]:
# sort cfihos classes by number of properties
cfihos_classes.sort(key=lambda x: len(x.properties), reverse=True)

In [21]:
rprint(cfihos_classes[15].model_dump())

{
    'name': 'rotary compressor',
    'id_': 'CFIHOS-30000552',
    'level': 3,
    'description': 'A positive displacement compressor in which compression displacement is effected by the 
positive action of rotating elements.',
    'background': '',
    'properties': {
        'COMPRESSIBILITY FACTOR Z AT INLET': {
            'name': 'compressibility factor Z at inlet',
            'id_': 'CFIHOS-40000060',
            'dtype': 'Number',
            'description': 'The compressibility (Z) factor for the process medium at inlet conditions',
            'uom_required': True,
            'uom_name': 'volume proportion',
            'presence': 'NotApplicable'
        },
        'DIRECTION OF ROTATION': {
            'name': 'direction of rotation',
            'id_': 'CFIHOS-40000075',
            'dtype': 'Text',
            'description': 'Specify the direction of rotation of the functional or physical object.',
            'uom_required': True,
            'uom_name': None,
            'presence': 'NotApplicable'
        },
        'EXPLOSION PROTECTION GAS GROUP': {
            'name': 'explosion protection gas group',
            'id_': 'CFIHOS-40000110',
            'dtype': 'Text',
            'description': 'Specify the gas group classification of the physical object in accordance with IEC/EN 
60079-20',
            'uom_required': True,
            'uom_name': None,
            'presence': 'NotApplicable'
        },
        'EXPLOSION PROTECTION TEMPERATURE CLASS': {
            'name': 'explosion protection temperature class',
            'id_': 'CFIHOS-40000112',
            'dtype': 'Text',
            'description': 'Specify the temperature classification of the physical object in accordance with 
CENELEC EN50014',
            'uom_required': True,
            'uom_name': None,
            'presence': 'NotApplicable'
        },
        'EXPLOSION PROTECTION ZONE': {
            'name': 'explosion protection zone',
            'id_': 'CFIHOS-40000113',
            'dtype': 'Text',
            'description': 'Specify the hazardous zone in which the functional or physical object is expected to 
operate.',
            'uom_required': True,
            'uom_name': None,
            'presence': 'NotApplicable'
        },
        'FLUID NAME': {
            'name': 'fluid name',
            'id_': 'CFIHOS-40000132',
            'dtype': 'Text',
            'description': 'The name of the product in liquid or gas phase.',
            'uom_required': True,
            'uom_name': None,
            'presence': 'NotApplicable'
        },
        'INSULATION CLASS': {
            'name': 'insulation class',
            'id_': 'CFIHOS-40000177',
            'dtype': 'Text',
            'description': 'The class of electrical insulation used by a  physical object.',
            'uom_required': True,
            'uom_name': None,
            'presence': 'NotApplicable'
        },
        'LOWER LIMIT OPERATING SUCTION PRESSURE': {
            'name': 'lower limit operating suction pressure',
            'id_': 'CFIHOS-40000219',
            'dtype': 'Number',
            'description': 'The suction pressure which is the lowest pressure at  which a functional or physical 
object is expected to operate.',
            'uom_required': True,
            'uom_name': 'pressure',
            'presence': 'NotApplicable'
        },
        'NORMAL OPERATING INLET TEMPERATURE': {
            'name': 'normal operating inlet temperature',
            'id_': 'CFIHOS-40000272',
            'dtype': 'Number',
            'description': 'The temperature at which a functional or physical object is expected to operate 
normally at its inlet connection.',
            'uom_required': True,
            'uom_name': 'temperature',
            'presence': 'NotApplicable'
        },
        'NORMAL OPERATING MASS FLOW RATE': {
            'name': 'normal operating mass flow rate',
            'id_': 'CFIHOS-40000275',
            'dtype': 'Number',
     

# Dump Results to File

In [22]:
out = folder / f"{output_file_name}.json"
with open(out, "w") as f:
    out = [c.model_dump_json(indent=2, by_alias=True) for c in cfihos_classes]
    f.write(f"[{','.join(out)}]")